## 환경 체크

In [ ]:
import torch, sys, subprocess

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")

# 필수 패키지 버전
for pkg in ["transformers", "datasets", "scikit-learn", "numpy", "scipy"]:
    try:
        m = __import__(pkg.replace("-", "_"))
        print(f"{pkg}: {m.__version__}")
    except Exception as e:
        print(f"{pkg}: NOT INSTALLED ({e})")

Python: 3.12.13
CUDA available: True
GPU: NVIDIA L4
VRAM: 23.7 GB
transformers: 5.13.1
datasets: 4.0.0
scikit-learn: NOT INSTALLED (No module named 'scikit_learn')
numpy: 2.0.2
scipy: 1.16.3


In [ ]:
!pip install -q scikit-learn umap-learn hdbscan

import sklearn, umap, hdbscan
print("scikit-learn:", sklearn.__version__)
print("umap-learn:", umap.__version__)

scikit-learn: 1.6.1
umap-learn: 0.5.12


In [ ]:
from huggingface_hub import login
login()

In [ ]:
# 클러스터링은 sklearn 내장으로 통일 (hdbscan 별도 패키지 불필요)
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
import umap  # 시각화 전용
print("clustering/metrics import OK")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
    attn_implementation="eager",   # attention weight 추출하려면 eager 필수 (sdpa는 weight 반환 안 함)
)
model.eval()
print("model loaded:", model.config.model_type, "| layers:", model.config.num_hidden_layers)

clustering/metrics import OK


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

model loaded: llama | layers: 32


## 데이터 세팅

### NQ Swap

In [ ]:
from datasets import load_dataset

# NQ-Swap: 정답 엔티티를 같은 타입 다른 엔티티로 치환한 반사실 문맥
ds = load_dataset("pminervini/NQ-Swap", split="dev")

print("rows:", len(ds))
print("columns:", ds.column_names)
print("\n--- 예시 1개 ---")
ex = ds[0]
for k, v in ex.items():
    s = str(v)
    print(f"[{k}] {s[:300]}{'...' if len(s)>300 else ''}")

rows: 4746
columns: ['question', 'org_context', 'org_answer', 'sub_context', 'sub_answer']

--- 예시 1개 ---
[question] how many episodes are in chicago fire season 4
[org_context] <P> The fourth season of Chicago Fire , an American drama television series with executive producer Dick Wolf , and producers Derek Haas , Michael Brandt , and Matt Olmstead , was ordered on February 5 , 2015 , by NBC , and premiered on October 13 , 2015 and concluded on May 17 , 2016 . The season c...
[org_answer] ['23']
[sub_context] <P> The fourth season of Chicago Fire , an American drama television series with executive producer Dick Wolf , and producers Derek Haas , Michael Brandt , and Matt Olmstead , was ordered on February 5 , 2015 , by NBC , and premiered on October 13 , 2015 and concluded on May 17 , 2016 . The season c...
[sub_answer] ['775']


In [ ]:
import re

# --- 통일 스키마 필드 (전 데이터셋 공통) ---
SCHEMA_FIELDS = [
    "sample_id", "dataset_id",
    "fact_nature", "construction_operator", "reasoning_structure", "provenance", "desired_action",
    "world_truth", "context_internal_validity", "benchmark_target",
    "question", "context", "parametric_answer", "contextual_answer",
    "relation_type", "answer_type", "hop_count", "domain",
    # K_state, model_action, final_answer, *_length, *_count 등은 이후 Phase에서 채움
]

def clean_ctx(t):
    return re.sub(r"\s+", " ", t.replace("<P>", "").replace("</P>", "")).strip()

def convert_nqswap(ex, idx):
    return {
        "sample_id": f"nqswap_{idx}",
        "dataset_id": "nqswap",
        "fact_nature": "static",
        "construction_operator": "entity_swap",
        "reasoning_structure": "single_hop",
        "provenance": "semi_synthetic",   # 자연 문서 + 엔티티 치환
        "desired_action": "accept",       # 벤치 원목표: 치환 문맥 추종
        "world_truth": "false",           # sub 기준 현실 거짓
        "context_internal_validity": "coherent",
        "benchmark_target": "contextual",
        "question": ex["question"].strip(),
        "context": clean_ctx(ex["sub_context"]),   # 반사실 문맥 사용
        "parametric_answer": ex["org_answer"][0] if ex["org_answer"] else None,
        "contextual_answer": ex["sub_answer"][0] if ex["sub_answer"] else None,
        "relation_type": None,
        "answer_type": None,
        "hop_count": 1,
        "domain": None,
    }

# 변환 테스트 (3개만)
for i in range(3):
    r = convert_nqswap(ds[i], i)
    print(r["sample_id"], "| Q:", r["question"])
    print("   param:", r["parametric_answer"], "| ctx:", r["contextual_answer"], "| action:", r["desired_action"])
    print("   ctx[:120]:", r["context"][:120])
    print()

nqswap_0 | Q: how many episodes are in chicago fire season 4
   param: 23 | ctx: 775 | action: accept
   ctx[:120]: The fourth season of Chicago Fire , an American drama television series with executive producer Dick Wolf , and producer

nqswap_1 | Q: how many episodes are in chicago fire season 4
   param: 23 | ctx: three | action: accept
   ctx[:120]: The fourth season of Chicago Fire , an American drama television series with executive producer Dick Wolf , and producer

nqswap_2 | Q: who sings love will keep us alive by the eagles
   param: Timothy B. Schmit | ctx: Yuliya Snigir | action: accept
   ctx[:120]: `` Love Will Keep Us Alive '' is a song written by Jim Capaldi , Paul Carrack , and Peter Vale , and produced by the Eag



### ClashEval

In [ ]:
from datasets import load_dataset

ds_clash = load_dataset("sagnikrayc/clasheval", split="train")

print("rows:", len(ds_clash))
print("columns:", ds_clash.column_names)
print("\n--- 예시 1개 ---")
ex = ds_clash[0]
for k, v in ex.items():
    s = str(v)
    print(f"[{k}] {s[:300]}{'...' if len(s)>300 else ''}")

README.md:   0%|          | 0.00/397 [00:00<?, ?B/s]

clasheval.parquet: reconstructing file:   0%|          |  0.00B / 25.3MB            

clasheval.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10179 [00:00<?, ? examples/s]

rows: 10179
columns: ['question', 'context_original', 'context_mod', 'answer_original', 'answer_mod', 'mod_degree', 'dataset']

--- 예시 1개 ---
[question] Which former United States Senator, born in 1955, also shares the surname with other senators at the state level in Wisconsin, Minnesota, Massachusetts, Puerto Rico, and New York City?
[context_original] Joe Donnelly (born 1955) is a former United States Senator from Indiana. Senator Donnelly may also refer to:

Davis A. Donnelly (1927–2020), Wisconsin State Senate
Ignatius L. Donnelly (1831–1901), Minnesota State Senate
Ken Donnelly (1950–2017), Massachusetts State Senate
Melinda Romero Donnelly (b...
[context_mod] Joey McJoeFace (born 1955) is a former United States Senator from Indiana. Senator Donnelly may also refer to:

Davis A. Donnelly (1927–2020), Wisconsin State Senate
Ignatius L. Donnelly (1831–1901), Minnesota State Senate
Ken Donnelly (1950–2017), Massachusetts State Senate
Melinda Romero Donnelly ...
[answer_original] Joe

In [ ]:
from collections import Counter

print("=== dataset(domain) 분포 ===")
for k, v in Counter(ds_clash["dataset"]).most_common():
    print(f"  {k}: {v}")

print("\n=== mod_degree 분포 ===")
for k, v in sorted(Counter(ds_clash["mod_degree"]).items()):
    print(f"  degree {k}: {v}")

=== dataset(domain) 분포 ===
  drugs: 2653
  news: 2416
  years: 2000
  records: 1910
  names: 600
  locations: 600

=== mod_degree 분포 ===
  degree -0.19999999999999996: 4
  degree -0.31000000000000005: 1
  degree -0.3599999999999999: 1
  degree -0.3999999999999999: 2
  degree -0.5: 3
  degree -0.6: 4
  degree -0.6000000000000001: 9
  degree -0.6100000000000001: 1
  degree -0.6499999999999999: 1
  degree -0.6999999999999997: 1
  degree -0.7999999999999998: 2
  degree -0.8: 4
  degree -0.8999999999999999: 1
  degree -0.9: 4
  degree -0.96: 1
  degree -1: 32
  degree -1.01: 1
  degree -1.2: 2
  degree -1.2000000000000002: 4
  degree -1.3599999999999999: 1
  degree -1.4100000000000001: 1
  degree -1.45: 1
  degree -1.5: 4
  degree -1.5599999999999998: 1
  degree -1.5999999999999996: 3
  degree -1.6: 2
  degree -1.61: 1
  degree -1.7000000000000002: 1
  degree -1.71: 1
  degree -1.799999999999999: 1
  degree -1.8: 11
  degree -1.85: 1
  degree -1.87: 1
  degree -1.8999999999999995: 1
  degre

In [ ]:
# domain → (answer_type, operator) 매핑
CLASH_DOMAIN_MAP = {
    "years":     ("numeric", "numeric_perturbation"),
    "records":   ("numeric", "numeric_perturbation"),
    "drugs":     ("numeric", "numeric_perturbation"),
    "news":      ("numeric", "numeric_perturbation"),
    "names":     ("entity",  "entity_swap"),
    "locations": ("entity",  "entity_swap"),
}

def bin_mod_degree(d):
    a = abs(float(d))
    if a <= 1:    return "tiny"     # ≤100% 변화
    if a <= 10:   return "small"
    if a <= 100:  return "medium"
    return "large"

def convert_clasheval(ex, idx):
    dom = ex["dataset"]
    ans_type, op = CLASH_DOMAIN_MAP.get(dom, ("other", "other"))
    return {
        "sample_id": f"clasheval_{idx}",
        "dataset_id": "clasheval",
        "fact_nature": "static",
        "construction_operator": op,
        "reasoning_structure": "single_hop",
        "provenance": "semi_synthetic",
        "desired_action": "reject",            # ← NQ-Swap과 정반대
        "world_truth": "false",                # mod 문맥은 현실 거짓
        "context_internal_validity": "coherent",
        "benchmark_target": "parametric",      # ← 자기 prior 유지가 정답
        "question": ex["question"].strip(),
        "context": ex["context_mod"].strip(),  # perturbed 문맥
        "parametric_answer": ex["answer_original"],  # 현실 참
        "contextual_answer": ex["answer_mod"],       # 거짓
        "relation_type": None,
        "answer_type": ans_type,
        "hop_count": 1,
        "domain": dom,
        "mod_degree_bin": bin_mod_degree(ex["mod_degree"]),  # confound
    }

# 테스트: 각 domain에서 1개씩
seen = set()
for i, ex in enumerate(ds_clash):
    if ex["dataset"] not in seen:
        seen.add(ex["dataset"])
        r = convert_clasheval(ex, i)
        print(f"[{r['domain']}] op={r['construction_operator']}, ans_type={r['answer_type']}, deg={r['mod_degree_bin']}")
        print(f"   Q: {r['question'][:80]}")
        print(f"   param(true): {r['parametric_answer']} | ctx(false): {r['contextual_answer']}")
        print()
    if len(seen) == 6:
        break

[names] op=entity_swap, ans_type=entity, deg=small
   Q: Which former United States Senator, born in 1955, also shares the surname with o
   param(true): Joe Donnelly | ctx(false): Joey McJoeFace

[locations] op=entity_swap, ans_type=entity, deg=tiny
   Q: What is the name of the hamlet in Canada that shares its name with a Scottish su
   param(true): Cassils | ctx(false): Brooks

[years] op=numeric_perturbation, ans_type=numeric, deg=medium
   Q: In which year was the census conducted that reported the population of Lukhi vil
   param(true): 2006 | ctx(false): 1973

[news] op=numeric_perturbation, ans_type=numeric, deg=medium
   Q: How many points did Paige Bueckers score in the Big East Tournament title game o
   param(true): 27 | ctx(false): 2

[drugs] op=numeric_perturbation, ans_type=numeric, deg=medium
   Q: What is the maximum daily dosage in mg for extended release oxybutynin in adults
   param(true): 30 | ctx(false): 3

[records] op=numeric_perturbation, ans_type=numeric, deg=

In [ ]:
import pandas as pd

# 1) 각 데이터셋 전체 변환
rows_nqswap = [convert_nqswap(ds[i], i) for i in range(len(ds))]
rows_clash  = [convert_clasheval(ds_clash[i], i) for i in range(len(ds_clash))]

# 2) DataFrame으로 (컬럼 자동 union, 없는 값은 NaN)
df = pd.DataFrame(rows_nqswap + rows_clash)

print("총 rows:", len(df))
print("\n=== dataset_id 분포 ===")
print(df["dataset_id"].value_counts())

print("\n=== 핵심 축 교차 확인 (dataset_id × desired_action) ===")
print(pd.crosstab(df["dataset_id"], df["desired_action"]))

print("\n=== construction_operator 분포 ===")
print(df["construction_operator"].value_counts())

print("\n=== 결측 필드 확인 ===")
print(df.isna().sum()[df.isna().sum() > 0])

총 rows: 14925

=== dataset_id 분포 ===
dataset_id
clasheval    10179
nqswap        4746
Name: count, dtype: int64

=== 핵심 축 교차 확인 (dataset_id × desired_action) ===
desired_action  accept  reject
dataset_id                    
clasheval            0   10179
nqswap            4746       0

=== construction_operator 분포 ===
construction_operator
numeric_perturbation    8979
entity_swap             5946
Name: count, dtype: int64

=== 결측 필드 확인 ===
relation_type     14925
answer_type        4746
domain             4746
mod_degree_bin     4746
dtype: int64


### ConFiQA 데이터

In [ ]:
from datasets import load_dataset, get_dataset_config_names

REPO = "osunlp/ConFiQA"   # 후보 경로
try:
    configs = get_dataset_config_names(REPO)
    print("configs:", configs)
except Exception as e:
    print("config 조회 실패:", e)
    print("→ 경로 직접 확인 필요")

config 조회 실패: Dataset 'osunlp/ConFiQA' doesn't exist on the Hub or cannot be accessed.
→ 경로 직접 확인 필요


In [ ]:
import urllib.request, json

url = "https://raw.githubusercontent.com/byronBBL/Context-DPO/master/ConFiQA/ConFiQA-QA.json"
urllib.request.urlretrieve(url, "ConFiQA-QA.json")

with open("ConFiQA-QA.json") as f:
    confiqa = json.load(f)

print("rows:", len(confiqa))
print("keys:", list(confiqa[0].keys()))
print("\n--- 예시 1개 ---")
ex = confiqa[0]
for k in ["question", "orig_answer", "cf_answer", "orig_path_labeled", "cf_path_labeled"]:
    print(f"[{k}] {ex[k]}")
print(f"[cf_context] {ex['cf_context'][:200]}...")

rows: 6000
keys: ['orig_path', 'cf_path', 'orig_path_labeled', 'cf_path_labeled', 'orig_triple', 'cf_triple', 'orig_answer', 'cf_answer', 'orig_alias', 'cf_alias', 'question', 'orig_context_piece', 'cf_context_piece', 'orig_context', 'cf_context']

--- 예시 1개 ---
[question] Who is the composer of Bad Boys for Life?
[orig_answer] Lorne Balfe
[cf_answer] Petri Alanko
[orig_path_labeled] [('Bad Boys for Life', 'composer', 'Lorne Balfe')]
[cf_path_labeled] [('Bad Boys for Life', 'composer', 'Petri Alanko')]
[cf_context] Bad Boys for Life is a 2020 action comedy film directed by Adil El Arbi and Bilall Fallah, and composed by Petri Alanko. The movie follows two old friends, Mike Lowrey (Will Smith) and Marcus Burnett ...


In [ ]:
import ast

def parse_relation(path_labeled):
    """[('Bad Boys for Life', 'composer', 'Petri Alanko')] → 'composer'"""
    try:
        parsed = ast.literal_eval(path_labeled)
        return parsed[0][1] if parsed and len(parsed[0]) >= 2 else None
    except Exception:
        return None

def convert_confiqa(ex, idx):
    return {
        "sample_id": f"confiqa_qa_{idx}",
        "dataset_id": "confiqa_qa",
        "fact_nature": "static",
        "construction_operator": "relation_edit",   # ← 새 operator
        "reasoning_structure": "single_hop",
        "provenance": "synthetic",                   # ChatGPT-4 생성 context
        "desired_action": "accept",                  # context-faithful = 문맥 추종
        "world_truth": "false",
        "context_internal_validity": "coherent",
        "benchmark_target": "contextual",
        "question": ex["question"].strip(),
        "context": ex["cf_context"].strip(),
        "parametric_answer": ex["orig_answer"],
        "contextual_answer": ex["cf_answer"],
        "relation_type": parse_relation(ex["cf_path_labeled"]),
        "answer_type": "entity",
        "hop_count": 1,
        "domain": None,
    }

# 테스트 3개
for i in range(3):
    r = convert_confiqa(confiqa[i], i)
    print(f"{r['sample_id']} | rel={r['relation_type']} | action={r['desired_action']}")
    print(f"   Q: {r['question']}")
    print(f"   param(true): {r['parametric_answer']} | ctx(false): {r['contextual_answer']}")
    print()

# relation_type 분포도 확인 (nesting 완화에 중요)
from collections import Counter
rels = [parse_relation(ex["cf_path_labeled"]) for ex in confiqa]
print("고유 relation 수:", len(set(rels)))
print("상위 10개:", Counter(rels).most_common(10))

confiqa_qa_0 | rel=composer | action=accept
   Q: Who is the composer of Bad Boys for Life?
   param(true): Lorne Balfe | ctx(false): Petri Alanko

confiqa_qa_1 | rel=country of origin | action=accept
   Q: What is the country of origin of the film Lightyear?
   param(true): United States of America | ctx(false): United Kingdom

confiqa_qa_2 | rel=chief executive officer | action=accept
   Q: Who is the chief executive officer of GB News?
   param(true): Angelos Frangopoulos | ctx(false): David Brandon

고유 relation 수: 40
상위 10개: [('award received', 587), ('country of citizenship', 575), ('genre', 437), ('spouse', 436), ('member of', 311), ('religion', 257), ('sport', 255), ('composer', 243), ('country of origin', 241), ('field of work', 208)]


In [ ]:
import pandas as pd

rows_confiqa = [convert_confiqa(confiqa[i], i) for i in range(len(confiqa))]
df = pd.DataFrame(rows_nqswap + rows_clash + rows_confiqa)

print("총 rows:", len(df))
print("\n=== dataset_id 분포 ===")
print(df["dataset_id"].value_counts())

print("\n=== 핵심: desired_action × dataset_id (nesting 확인) ===")
print(pd.crosstab(df["desired_action"], df["dataset_id"]))

print("\n=== operator × desired_action ===")
print(pd.crosstab(df["construction_operator"], df["desired_action"]))

총 rows: 20925

=== dataset_id 분포 ===
dataset_id
clasheval     10179
confiqa_qa     6000
nqswap         4746
Name: count, dtype: int64

=== 핵심: desired_action × dataset_id (nesting 확인) ===
dataset_id      clasheval  confiqa_qa  nqswap
desired_action                               
accept                  0        6000    4746
reject              10179           0       0

=== operator × desired_action ===
desired_action         accept  reject
construction_operator                
entity_swap              4746    1200
numeric_perturbation        0    8979
relation_edit            6000       0


### DynamicQA 데이터

In [ ]:
from datasets import load_dataset, get_dataset_config_names

REPO = "copenlu/dynamicqa"   # 후보 경로
try:
    print("configs:", get_dataset_config_names(REPO))
except Exception as e:
    print("config 조회 실패:", e)
    # 대체 시도
    for alt in ["kortukov/DynamicQA", "dynamicqa"]:
        try:
            print(f"  try {alt}:", get_dataset_config_names(alt))
        except Exception as e2:
            print(f"  {alt} 실패:", str(e2)[:80])

README.md:   0%|          | 0.00/2.46k [00:00<?, ?B/s]

configs: ['static', 'temporal', 'disputable']


In [ ]:
from datasets import load_dataset

dyn = {}
for cfg in ["static", "temporal", "disputable"]:
    d = load_dataset("copenlu/dynamicqa", cfg)
    split = list(d.keys())[0]   # split 이름 자동
    dyn[cfg] = d[split]
    print(f"[{cfg}] split={split}, rows={len(dyn[cfg])}, cols={dyn[cfg].column_names}")

print("\n--- temporal 예시 1개 ---")
ex = dyn["temporal"][0]
for k, v in ex.items():
    s = str(v)
    print(f"[{k}] {s[:250]}{'...' if len(s)>250 else ''}")

static.csv:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

[static] split=test, rows=2500, cols=['id', 'subj', 'question', 'context', 'obj', 'replace_name', 'num_edits', 's_pop']


temporal.csv:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2495 [00:00<?, ? examples/s]

[temporal] split=test, rows=2495, cols=['id', 'subj', 'question', 'context', 'obj', 'replace_name', 'num_edits', 's_pop']


disputable.csv:   0%|          | 0.00/198k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/694 [00:00<?, ? examples/s]

[disputable] split=test, rows=694, cols=['id', 'subj', 'question', 'context', 'obj', 'replace_name', 'num_edits', 's_pop']

--- temporal 예시 1개 ---
[id] 4222362
[subj] George Rankin
[question] What is George Rankin's occupation?
[context]  George James Rankin,  (1 May 1887 – 28 December 1957) was a [ENTITY]. He served in both the House of Representatives and the Senate, representing the Country Party of Australia.
[obj] politician
[replace_name] judge
[num_edits] 4.0
[s_pop] 142


In [ ]:
def fill_entity(context, filler):
    return context.replace("[ENTITY]", filler).strip()

for cfg in ["static", "temporal", "disputable"]:
    ex = dyn[cfg][0]
    print(f"=== [{cfg}] ===")
    print(f"  subj: {ex['subj']} | Q: {ex['question']}")
    print(f"  obj(param/true): {ex['obj']} | replace_name(counter): {ex['replace_name']}")
    print(f"  num_edits: {ex['num_edits']} | s_pop: {ex['s_pop']}")
    print(f"  [ENTITY] 포함?: {'[ENTITY]' in ex['context']}")
    print(f"  counter context: {fill_entity(ex['context'], ex['replace_name'])[:180]}")
    print()

=== [static] ===
  subj: Herlyn Espinal | Q: What is Herlyn Espinal's occupation?
  obj(param/true): journalist | replace_name(counter): literary critic
  num_edits: 1.0 | s_pop: 88
  [ENTITY] 포함?: True
  counter context: Herlyn Iván Espinal Martínez (14 September 1982 – 20 July 2014) was a literary critic.

=== [temporal] ===
  subj: George Rankin | Q: What is George Rankin's occupation?
  obj(param/true): politician | replace_name(counter): judge
  num_edits: 4.0 | s_pop: 142
  [ENTITY] 포함?: True
  counter context: George James Rankin,  (1 May 1887 – 28 December 1957) was a judge. He served in both the House of Representatives and the Senate, representing the Country Party of Australia.

=== [disputable] ===
  subj: Sean Hannity | Q: Which honorary degrees has Sean Hannity received from Liberty University and Southeastern University of Lakeland, Florida?
  obj(param/true): certificates | replace_name(counter): doctorates
  num_edits: 1 | s_pop: 68529
  [ENTITY] 포함?: True
  counter co

In [ ]:
DYN_MAP = {
    "static":     ("static",          "false",               "no_perturbation"),
    "temporal":   ("temporal_mutable", "outdated",           "temporal_replacement"),
    "disputable": ("disputable",       "viewpoint_dependent", "no_perturbation"),
}

def convert_dynamicqa(ex, cfg, idx):
    fact_nature, world_truth, op = DYN_MAP[cfg]
    return {
        "sample_id": f"dynamicqa_{cfg}_{idx}",
        "dataset_id": "dynamicqa",              # ← 단일 dataset_id (within-source 앵커 핵심)
        "fact_nature": fact_nature,             # ← 여기서 3종으로 갈림
        "construction_operator": op,
        "reasoning_structure": "single_hop",
        "provenance": "semi_synthetic",
        "desired_action": "needs_verification", # ← 억지 accept/reject 안 함
        "world_truth": world_truth,
        "context_internal_validity": "coherent",
        "benchmark_target": "abstention",
        "question": ex["question"].strip(),
        "context": ex["context"].replace("[ENTITY]", ex["replace_name"]).strip(),  # counter-memory
        "parametric_answer": ex["obj"],         # 원래 사실
        "contextual_answer": ex["replace_name"],# 치환된 문맥 답
        "relation_type": None,
        "answer_type": "entity",
        "hop_count": 1,
        "domain": cfg,                          # static/temporal/disputable 재기록
        "num_edits": float(ex["num_edits"]),    # dynamicity confound
        "s_pop": int(ex["s_pop"]),              # popularity confound (K-state 상관)
    }

# 세 subset 전체 변환 + 검증
rows_dyn = []
for cfg in ["static", "temporal", "disputable"]:
    rows_dyn += [convert_dynamicqa(dyn[cfg][i], cfg, i) for i in range(len(dyn[cfg]))]

df_dyn = pd.DataFrame(rows_dyn)
print("DynamicQA rows:", len(df_dyn))
print("\n=== fact_nature 분포 (within-source에서 이게 갈려야 함) ===")
print(df_dyn["fact_nature"].value_counts())
print("\n=== fact_nature × num_edits 평균 (temporal일수록 커야 정상) ===")
print(df_dyn.groupby("fact_nature")["num_edits"].mean())

DynamicQA rows: 5689

=== fact_nature 분포 (within-source에서 이게 갈려야 함) ===
fact_nature
static              2500
temporal_mutable    2495
disputable           694
Name: count, dtype: int64

=== fact_nature × num_edits 평균 (temporal일수록 커야 정상) ===
fact_nature
disputable          2.573487
static              1.000000
temporal_mutable    3.222044
Name: num_edits, dtype: float64


### Mulan 데이터

In [ ]:
from datasets import load_dataset, get_dataset_config_names

REPO = "coastalcph/fm_queries"
try:
    print("configs:", get_dataset_config_names(REPO))
except Exception as e:
    print("config 없음(단일):", str(e)[:60])

ds_mulan = load_dataset(REPO)
split = list(ds_mulan.keys())[0]
ds_mulan = ds_mulan[split]
print(f"\nsplit={split}, rows={len(ds_mulan)}")
print("columns:", ds_mulan.column_names)
print("\n--- 예시 2개 ---")
for i in range(2):
    ex = ds_mulan[i]
    for k, v in ex.items():
        s = str(v)
        print(f"  [{k}] {s[:150]}{'...' if len(s)>150 else ''}")
    print()

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

configs: ['default']


Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

P138.jsonl:   0%|          | 0.00/1.49M [00:00<?, ?B/s]

P1037.jsonl:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

...

P69.jsonl:   0%|          | 0.00/2.35M [00:00<?, ?B/s]

P740.jsonl:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

P937.jsonl:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/245710 [00:00<?, ? examples/s]


split=train, rows=245710
columns: ['query', 'answer', 'id', 'relation', 'date', 'type']

--- 예시 2개 ---
  [query] Barack Obama works in the field of _X_.
  [answer] [{'wikidata_id': 'Q25447176', 'name': 'civil rights'}, {'wikidata_id': 'Q11206', 'name': 'constitutional law'}]
  [id] Q76_P101_0
  [relation] P101
  [date] 2021
  [type] immutable_n

  [query] Barack Obama specializes in _X_.
  [answer] [{'wikidata_id': 'Q25447176', 'name': 'civil rights'}, {'wikidata_id': 'Q11206', 'name': 'constitutional law'}]
  [id] Q76_P101_1
  [relation] P101
  [date] 2021
  [type] immutable_n



In [ ]:
MULAN_NATURE = {
    "mutable":     "temporal_mutable",
    "immutable_1": "immutable",
    "immutable_n": "immutable",
}

def convert_mulan(ex, idx):
    ans = ex["answer"]
    ans_name = ans[0]["name"] if isinstance(ans, list) and ans else str(ans)
    return {
        "sample_id": f"mulan_{idx}",
        "dataset_id": "mulan",
        "fact_nature": MULAN_NATURE.get(ex["type"], "unknown"),
        "construction_operator": "no_perturbation",  # closed-book, 조작 없음
        "reasoning_structure": "single_hop",
        "provenance": "natural",
        "desired_action": "closed_book",   # context 없음 → 별도 트랙 표시
        "world_truth": "true",
        "context_internal_validity": None, # context 없음
        "benchmark_target": "parametric",
        "question": ex["query"].replace("_X_", "___").strip(),
        "context": None,                   # ← context 없음
        "parametric_answer": ans_name,
        "contextual_answer": None,
        "relation_type": ex["relation"],
        "answer_type": "entity",
        "hop_count": 1,
        "domain": ex["type"],              # mutable/immutable_1/immutable_n 원본 보존
        "mulan_date": ex["date"],
    }

# type 분포 확인 (전체 245k)
from collections import Counter
type_dist = Counter(ds_mulan["type"])
print("=== MuLan type 분포 ===")
for k, v in type_dist.most_common():
    print(f"  {k}: {v}")

# 매핑 테스트
print("\n--- 매핑 예시 3개 ---")
for i in range(0, 245710, 80000):
    r = convert_mulan(ds_mulan[i], i)
    print(f"  {r['sample_id']} | type={r['domain']}→{r['fact_nature']} | Q: {r['question'][:60]}")
    print(f"     ans: {r['parametric_answer']} | rel: {r['relation_type']}")

=== MuLan type 분포 ===
  mutable: 88680
  immutable: 88585
  immutable_n: 68445

--- 매핑 예시 3개 ---
  mulan_0 | type=immutable_n→immutable | Q: Barack Obama works in the field of ___.
     ans: civil rights | rel: P101
  mulan_80000 | type=immutable→unknown | Q: The headquarters of Tivoli Gardens is in ___ .
     ans: Copenhagen | rel: P159
  mulan_160000 | type=mutable→temporal_mutable | Q: Giuseppe Piazzi has the position of ___.
     ans: professor | rel: P39
  mulan_240000 | type=mutable→temporal_mutable | Q: Ursula von der Leyen works in the location of ___.
     ans: Hanover | rel: P937


In [ ]:
MULAN_NATURE = {
    "mutable":     "temporal_mutable",
    "immutable":   "immutable",     # ← 실제 값에 맞춤 (immutable_1 아님)
    "immutable_n": "immutable",
}

def convert_mulan(ex, idx):
    ans = ex["answer"]
    ans_name = ans[0]["name"] if isinstance(ans, list) and ans else str(ans)
    return {
        "sample_id": f"mulan_{idx}",
        "dataset_id": "mulan",
        "fact_nature": MULAN_NATURE.get(ex["type"], "unknown"),
        "construction_operator": "no_perturbation",
        "reasoning_structure": "single_hop",
        "provenance": "natural",
        "desired_action": "closed_book",
        "world_truth": "true",
        "context_internal_validity": None,
        "benchmark_target": "parametric",
        "question": ex["query"].replace("_X_", "___").strip(),
        "context": None,
        "parametric_answer": ans_name,
        "contextual_answer": None,
        "relation_type": ex["relation"],
        "answer_type": "entity",
        "hop_count": 1,
        "domain": ex["type"],
        "mulan_date": ex["date"],
    }

# unknown 안 남는지 검증
sample_natures = Counter(convert_mulan(ds_mulan[i], i)["fact_nature"]
                         for i in range(0, 245710, 500))
print("fact_nature 분포(샘플):", dict(sample_natures))
assert "unknown" not in sample_natures, "아직 unknown 남음!"
print("✓ unknown 없음")

# mutability 재현 검증용 균형 확인
print("\ntemporal_mutable(=mutable):", type_dist["mutable"])
print("immutable(immutable + immutable_n):", type_dist["immutable"] + type_dist["immutable_n"])

fact_nature 분포(샘플): {'immutable': 315, 'temporal_mutable': 177}
✓ unknown 없음

temporal_mutable(=mutable): 88680
immutable(immutable + immutable_n): 157030


In [ ]:
import numpy as np
np.random.seed(42)

# MuLan 다운샘플: type별 균등하게 총 ~6000개 (검증 트랙엔 충분)
mulan_idx_by_type = {"mutable": [], "immutable": [], "immutable_n": []}
for i in range(len(ds_mulan)):
    mulan_idx_by_type[ds_mulan[i]["type"]].append(i)

per_type = 2000
mulan_keep = []
for t, idxs in mulan_idx_by_type.items():
    chosen = np.random.choice(idxs, size=min(per_type, len(idxs)), replace=False)
    mulan_keep.extend(chosen.tolist())

rows_mulan = [convert_mulan(ds_mulan[i], i) for i in mulan_keep]
print(f"MuLan 다운샘플: {len(rows_mulan)}개")

# === 5개 전체 통합 ===
df_all = pd.DataFrame(rows_nqswap + rows_clash + rows_confiqa + rows_dyn + rows_mulan)

# 컬럼 순서 정리 (스키마 필드 우선)
print("\n총 rows:", len(df_all))
print("\n=== dataset_id × desired_action ===")
print(pd.crosstab(df_all["dataset_id"], df_all["desired_action"]))
print("\n=== fact_nature × dataset_id ===")
print(pd.crosstab(df_all["fact_nature"], df_all["dataset_id"]))

# 저장
df_all.to_parquet("conflict_pool_raw.parquet", index=False)
print("\n✓ 저장: conflict_pool_raw.parquet")
print("컬럼:", list(df_all.columns))

MuLan 다운샘플: 6000개

총 rows: 32614

=== dataset_id × desired_action ===
desired_action  accept  closed_book  needs_verification  reject
dataset_id                                                     
clasheval            0            0                   0   10179
confiqa_qa        6000            0                   0       0
dynamicqa            0            0                5689       0
mulan                0         6000                   0       0
nqswap            4746            0                   0       0

=== fact_nature × dataset_id ===
dataset_id        clasheval  confiqa_qa  dynamicqa  mulan  nqswap
fact_nature                                                      
disputable                0           0        694      0       0
immutable                 0           0          0   4000       0
static                10179        6000       2500      0    4746
temporal_mutable          0           0       2495   2000       0

✓ 저장: conflict_pool_raw.parquet
컬럼: ['sample_id', '

### Step 1-P4a: candidates 포함 저장 + 다운로드

In [ ]:
# answer_candidates 리스트가 parquet에 잘 저장되는지 확인하며 저장
df_all.to_parquet("conflict_pool_with_candidates.parquet", index=False)

# 검증: 다시 읽어서 candidates가 리스트로 살아있는지
import pandas as pd
chk = pd.read_parquet("conflict_pool_with_candidates.parquet")
print("rows:", len(chk))
print("candidates 타입:", type(chk["answer_candidates"].iloc[0]))
print("예시:", chk["answer_candidates"].iloc[0])
print("컬럼:", list(chk.columns))

# Colab에서 다운로드
from google.colab import files
files.download("conflict_pool_with_candidates.parquet")

rows: 32614
candidates 타입: <class 'numpy.ndarray'>
예시: ['23']
컬럼: ['sample_id', 'dataset_id', 'fact_nature', 'construction_operator', 'reasoning_structure', 'provenance', 'desired_action', 'world_truth', 'context_internal_validity', 'benchmark_target', 'question', 'context', 'parametric_answer', 'contextual_answer', 'relation_type', 'answer_type', 'hop_count', 'domain', 'mod_degree_bin', 'num_edits', 's_pop', 'mulan_date', 'answer_candidates']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 이 아래부터는 A100 세션으로 전환해서 수행. 의미 없는 셀

## Phase 1 _ Closed book

### EM 판정 규칙

In [ ]:
# 각 데이터셋 alias 사용 가능 여부 확인
print("=== alias 가용성 ===")
print("ConFiQA: orig_alias 있음 →", "orig_alias" in confiqa[0])
print("  예시:", confiqa[0].get("orig_alias"))
print("MuLan: answer가 복수 정답 리스트 →", ds_mulan[0]["answer"][:2])
print("NQ-Swap: org_answer가 리스트 →", ds[0]["org_answer"])
print("ClashEval: answer_original 단일 →", ds_clash[0]["answer_original"])
print("DynamicQA: obj 단일 →", dyn['static'][0]["obj"])

=== alias 가용성 ===
ConFiQA: orig_alias 있음 → True
  예시: ['Lorne David Roderick Balfe']
MuLan: answer가 복수 정답 리스트 → [{'wikidata_id': 'Q25447176', 'name': 'civil rights'}, {'wikidata_id': 'Q11206', 'name': 'constitutional law'}]
NQ-Swap: org_answer가 리스트 → ['23']
ClashEval: answer_original 단일 → Joe Donnelly
DynamicQA: obj 단일 → journalist


In [ ]:
# sample_id → answer_candidates(현실 정답 후보군) 매핑 구축
cand = {}

# NQ-Swap: org_answer 리스트 그대로
for i in range(len(ds)):
    cand[f"nqswap_{i}"] = [str(a) for a in ds[i]["org_answer"]]

# ClashEval: answer_original 단일
for i in range(len(ds_clash)):
    cand[f"clasheval_{i}"] = [str(ds_clash[i]["answer_original"])]

# ConFiQA: orig_answer + orig_alias
for i in range(len(confiqa)):
    c = [confiqa[i]["orig_answer"]] + list(confiqa[i].get("orig_alias", []))
    cand[f"confiqa_qa_{i}"] = [str(x) for x in c]

# DynamicQA: obj 단일
for cfg in ["static", "temporal", "disputable"]:
    for i in range(len(dyn[cfg])):
        cand[f"dynamicqa_{cfg}_{i}"] = [str(dyn[cfg][i]["obj"])]

# MuLan: answer 리스트 전체 name (복수 정답)
for i in mulan_keep:
    names = [a["name"] for a in ds_mulan[i]["answer"] if isinstance(a, dict)]
    cand[f"mulan_{i}"] = names if names else [str(ds_mulan[i]["answer"])]

# df_all에 머지
df_all["answer_candidates"] = df_all["sample_id"].map(cand)

# 검증
missing = df_all["answer_candidates"].isna().sum()
print("후보군 결측:", missing)
print("\n샘플별 후보군 개수 분포:")
print(df_all["answer_candidates"].apply(lambda x: len(x) if isinstance(x, list) else 0).describe())
print("\n--- 데이터셋별 예시 ---")
for did in df_all["dataset_id"].unique():
    row = df_all[df_all["dataset_id"]==did].iloc[0]
    print(f"  [{did}] param={row['parametric_answer']} | candidates={row['answer_candidates'][:3]}")

후보군 결측: 0

샘플별 후보군 개수 분포:
count    32614.000000
mean         2.297326
std          4.088719
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        104.000000
Name: answer_candidates, dtype: float64

--- 데이터셋별 예시 ---
  [nqswap] param=23 | candidates=['23']
  [clasheval] param=Joe Donnelly | candidates=['Joe Donnelly']
  [confiqa_qa] param=Lorne Balfe | candidates=['Lorne Balfe', 'Lorne David Roderick Balfe']
  [dynamicqa] param=journalist | candidates=['journalist']
  [mulan] param=FC Sochaux-Montbéliard | candidates=['FC Sochaux-Montbéliard', 'Club Brugge K.V.', 'Borussia Dortmund']


In [ ]:
import re, string

def normalize_answer(s):
    """표준 QA 정규화: 소문자, 관사·구두점 제거, 공백 정규화"""
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def is_known(generation, candidates):
    """closed-book 생성물에 정답 후보 중 하나라도 포함되면 K+ (substring, 관대)"""
    gen_norm = normalize_answer(generation)
    if not gen_norm:
        return False
    for c in candidates:
        c_norm = normalize_answer(c)
        if c_norm and c_norm in gen_norm:   # 정답이 생성물의 substring
            return True
    return False

# 함수 검증 (엣지 케이스)
tests = [
    ("The answer is Barack Obama, the 44th president.", ["Barack Obama"], True),
    ("Obama", ["Barack Obama"], False),                    # 역방향 substring은 불인정
    ("Lorne David Roderick Balfe composed it", ["Lorne Balfe", "Lorne David Roderick Balfe"], True),  # alias로 잡힘
    ("I think it's a journalist by profession", ["journalist"], True),
    ("politician", ["judge"], False),                       # 오답
    ("23 episodes", ["23"], True),
]
print("=== EM 함수 검증 ===")
for gen, cand_list, expected in tests:
    got = is_known(gen, cand_list)
    mark = "✓" if got == expected else "✗ FAIL"
    print(f"  {mark} is_known({gen[:40]!r}, {cand_list}) = {got} (기대: {expected})")

=== EM 함수 검증 ===
  ✓ is_known('The answer is Barack Obama, the 44th pre', ['Barack Obama']) = True (기대: True)
  ✓ is_known('Obama', ['Barack Obama']) = False (기대: False)
  ✓ is_known('Lorne David Roderick Balfe composed it', ['Lorne Balfe', 'Lorne David Roderick Balfe']) = True (기대: True)
  ✓ is_known("I think it's a journalist by profession", ['journalist']) = True (기대: True)
  ✓ is_known('politician', ['judge']) = False (기대: False)
  ✓ is_known('23 episodes', ['23']) = True (기대: True)


### closed book 배치 추론

In [ ]:
import torch
from tqdm.auto import tqdm

def build_closedbook_prompt(question):
    # context 없음! 질문만. Llama-3.1 chat 템플릿
    msgs = [
        {"role": "system", "content": "Answer the question concisely with only the answer, no explanation."},
        {"role": "user", "content": f"Question: {question}\nAnswer:"},
    ]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def greedy_batch(questions, batch_size=16, max_new=32):
    outs = []
    for i in tqdm(range(0, len(questions), batch_size)):
        batch = [build_closedbook_prompt(q) for q in questions[i:i+batch_size]]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=512).to("cuda")
        gen = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                             pad_token_id=tok.eos_token_id)
        for j in range(len(batch)):
            new_tokens = gen[j][enc["input_ids"].shape[1]:]
            outs.append(tok.decode(new_tokens, skip_special_tokens=True).strip())
    return outs

# padding 설정 (Llama는 기본 pad 없음)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "left"   # decoder-only 생성은 left padding

# === 100개 파일럿 ===
pilot = df_all.sample(100, random_state=42).reset_index(drop=True)
pilot_gens = greedy_batch(pilot["question"].tolist())
pilot["cb_generation"] = pilot_gens
pilot["K_state"] = [
    "K+" if is_known(g, c) else "K-"
    for g, c in zip(pilot_gens, pilot["answer_candidates"])
]

print("=== 100개 파일럿 K-state 분포 ===")
print(pilot["K_state"].value_counts())
print("\n=== 생성 예시 5개 ===")
for _, r in pilot.head(5).iterrows():
    print(f"[{r['K_state']}] Q: {r['question'][:55]}")
    print(f"   gen: {r['cb_generation'][:70]!r}")
    print(f"   param: {r['parametric_answer']}")
    print()

  0%|          | 0/7 [00:00<?, ?it/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


=== 100개 파일럿 K-state 분포 ===
K_state
K-    62
K+    38
Name: count, dtype: int64

=== 생성 예시 5개 ===
[K-] Q: The person who assumes the role of head coach for FC Sl
   gen: "I couldn't find any information on the current head coach of FC Slavia"
   param: Yury Maleyew

[K-] Q: What is the Olympic record for Men's 216 arrow team ran
   gen: "I couldn't find any information on the Olympic record for Men's 216 ar"
   param: 2087

[K-] Q: Lady Gaga is originally from ___.
   gen: 'New York City.'
   param: Lenox Hill Hospital

[K-] Q: Where did Clark Gable work?
   gen: 'MGM Studios.'
   param: United States of America

[K+] Q: Tipu Sultan follows the faith of ___.
   gen: 'Islam'
   param: Islam



In [ ]:
REFUSAL_PATTERNS = [
    "couldn't find", "could not find", "don't have", "do not have",
    "no information", "not sure", "i'm not aware", "i am not aware",
    "unable to", "cannot find", "can't find", "i don't know", "i do not know",
    "there is no", "not enough information", "insufficient",
]

def detect_refusal(gen):
    g = gen.lower()
    return any(p in g for p in REFUSAL_PATTERNS)

def classify_kstate(gen, candidates):
    """
    K+           : 정답 후보 포함 (안다)
    K-_refusal   : 모른다고 명시 (진짜 모름)
    K-_other     : 다른 사실을 답함 (경계 — instability로 재검증 필요)
    """
    if is_known(gen, candidates):
        return "K+"
    if detect_refusal(gen):
        return "K-_refusal"
    return "K-_other"

# 파일럿에 재적용해서 3분류 확인
pilot["K_state3"] = [classify_kstate(g, c)
                     for g, c in zip(pilot["cb_generation"], pilot["answer_candidates"])]

print("=== 100개 파일럿 3분류 ===")
print(pilot["K_state3"].value_counts())

print("\n=== K-_other 예시 (경계 케이스 — 모델은 답했는데 gold와 다름) ===")
for _, r in pilot[pilot["K_state3"]=="K-_other"].head(6).iterrows():
    print(f"  Q: {r['question'][:50]}")
    print(f"     gen: {r['cb_generation'][:55]!r} | gold: {r['parametric_answer']}")

print("\n=== K-_refusal 예시 (진짜 모름) ===")
for _, r in pilot[pilot["K_state3"]=="K-_refusal"].head(3).iterrows():
    print(f"  gen: {r['cb_generation'][:60]!r}")

=== 100개 파일럿 3분류 ===
K_state3
K-_other      50
K+            38
K-_refusal    12
Name: count, dtype: int64

=== K-_other 예시 (경계 케이스 — 모델은 답했는데 gold와 다름) ===
  Q: Lady Gaga is originally from ___.
     gen: 'New York City.' | gold: Lenox Hill Hospital
  Q: Where did Clark Gable work?
     gen: 'MGM Studios.' | gold: United States of America
  Q: What is the Olympic record for Men's Clean & Jerk 
     gen: 'Lasha Talakhadze (Georgia) with 253 kg.' | gold: 265
  Q: The creative piece Johann Christoph Gottsched belo
     gen: 'Literary Criticism' | gold: theatre
  Q: Who was the composer of Let It Be?
     gen: 'Paul McCartney' | gold: John Lennon
  Q: Province of Ragusa is governed from the capital ci
     gen: 'Dubrovnik.' | gold: Ragusa

=== K-_refusal 예시 (진짜 모름) ===
  gen: "I couldn't find any information on the current head coach of"
  gen: "I couldn't find any information on the Olympic record for Me"
  gen: 'I couldn\'t find any information on a book called "Serpent Ma'
